# Lab 3.1: Time-Series Data Import and Missing Timestamp Analysis

**Name:** Zubair Moeen  
**Reg Number:** 22jzele0463  
**Lab:** Machine Learning Lab  
**Lab Supervisor:** Engr.Irshad Ullah  
**University:** UET Peshawar - Campus Nowshera 
## Lab Overview
This notebook introduces time-series data handling using the AEP hourly power dataset. The code imports the dataset, inspects its structure, checks date ranges, and identifies missing timestamp positions.

## Learning Objectives
- Import required data analysis and visualization libraries.
- Load hourly time-series data using Pandas.
- Inspect dataset rows, columns, data types, and summary statistics.
- Understand timestamp continuity and identify missing time records.

## Section 1: Library Import and Dataset Loading
This section imports Pandas, NumPy, Matplotlib, and Seaborn, then loads the AEP hourly dataset for time-series analysis.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [3]:
df=pd.read_csv(r'Z:\University\8th Semester\ML Lab\AEP_hourly.csv',parse_dates=True)

In [4]:
df.head()

,Datetime,AEP_MW
0,2004-12-31 01:00:00,13478.0
1,2004-12-31 02:00:00,12865.0
2,2004-12-31 03:00:00,12577.0
3,2004-12-31 04:00:00,12517.0
4,2004-12-31 05:00:00,12670.0


In [5]:
df.tail()

,Datetime,AEP_MW
121268,2018-01-01 20:00:00,21089.0
121269,2018-01-01 21:00:00,20999.0
121270,2018-01-01 22:00:00,20820.0
121271,2018-01-01 23:00:00,20415.0
121272,2018-01-02 00:00:00,19993.0


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  str    
 1   AEP_MW    121273 non-null  float64
dtypes: float64(1), str(1)
memory usage: 1.9 MB


In [7]:
df.describe()

,AEP_MW
count,121273.000000
mean,15499.513717
std,2591.399065
min,9581.000000
25%,13630.000000
50%,15310.000000
75%,17200.000000
max,25695.000000


### Dataset info
Oct 2004 to Aug 2018

In [8]:
# 2004 at indx 2207, 01/10/2004 23:00
# -1 is due to the missing of the first element of 2004 october
(24*31*2)+24*30 -1

2207

In [9]:
# 2018
# at index 116162, 02/08/2018 23:00
(24*31*4)+(24*30*2)+24*28+24*2+1

5137

## Section 2: Dataset Inspection
The following cells use head, tail, info, and describe to understand the dataset before applying any cleaning operation.


In [10]:
total = ((24*31*2)+24*30)-1 +  (24*31*7)*13+(24*30*4)*13+(24*28*1)*10+(24*29*1)*3  +  (24*31*4)+(24*30*2)+24*28+24*2+1
total

121296

In [11]:
print('Missing= ',  total - len(df))

Missing=  23


In [12]:
print('Data Points= ',  len(df))
print('Samples= ',      int(len(df)/24))

Data Points=  121273
Samples=  5053


In [13]:
df['Datetime']=pd.to_datetime(df['Datetime'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   Datetime  121273 non-null  datetime64[us]
 1   AEP_MW    121273 non-null  float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 1.9 MB


In [14]:
# Drop Duplicates Except the First Occurrence
df.drop_duplicates(subset=['Datetime'], keep ='first',  inplace= True)
print('len = ',len(df))

len =  121269


In [15]:
df

,Datetime,AEP_MW
0,2004-12-31 01:00:00,13478.0
1,2004-12-31 02:00:00,12865.0
2,2004-12-31 03:00:00,12577.0
3,2004-12-31 04:00:00,12517.0
4,2004-12-31 05:00:00,12670.0
...,...,...
121268,2018-01-01 20:00:00,21089.0
121269,2018-01-01 21:00:00,20999.0
121270,2018-01-01 22:00:00,20820.0
121271,2018-01-01 23:00:00,20415.0


In [16]:
df.set_index('Datetime', inplace=True)
df.head()

,AEP_MW
Datetime,
2004-12-31 01:00:00,13478.0
2004-12-31 02:00:00,12865.0
2004-12-31 03:00:00,12577.0
2004-12-31 04:00:00,12517.0
2004-12-31 05:00:00,12670.0


In [17]:
df.iloc[22:25]

,AEP_MW
Datetime,
2004-12-31 23:00:00,13478.0
2005-01-01 00:00:00,12892.0
2004-12-30 01:00:00,14097.0


In [18]:
df=df.sort_index(ascending=True)
df.iloc[22:25]

,AEP_MW
Datetime,
2004-10-01 23:00:00,14067.0
2004-10-02 00:00:00,13147.0
2004-10-02 01:00:00,12260.0


In [19]:
print(df.head(3))
print(df.tail(3))

                      AEP_MW
Datetime                    
2004-10-01 01:00:00  12379.0
2004-10-01 02:00:00  11935.0
2004-10-01 03:00:00  11692.0
                      AEP_MW
Datetime                    
2018-08-02 22:00:00  17001.0
2018-08-02 23:00:00  15964.0
2018-08-03 00:00:00  14809.0


## Section 3: Missing Timestamp Analysis
This section focuses on checking the date range and identifying where time values are missing in the hourly sequence.


In [20]:
missing_Timestamp = pd.date_range(
    '2004-10-01',
    '2018-08-03',
    freq='h'
).difference(df.index)

print('\nNumber of Missing Timestamp= ', len(missing_Timestamp), '\n')
print(missing_Timestamp.to_list())


Number of Missing Timestamp=  28 

[Timestamp('2004-10-01 00:00:00'), Timestamp('2004-10-31 02:00:00'), Timestamp('2005-04-03 03:00:00'), Timestamp('2005-10-30 02:00:00'), Timestamp('2006-04-02 03:00:00'), Timestamp('2006-10-29 02:00:00'), Timestamp('2007-03-11 03:00:00'), Timestamp('2007-11-04 02:00:00'), Timestamp('2008-03-09 03:00:00'), Timestamp('2008-11-02 02:00:00'), Timestamp('2009-03-08 03:00:00'), Timestamp('2009-11-01 02:00:00'), Timestamp('2010-03-14 03:00:00'), Timestamp('2010-11-07 02:00:00'), Timestamp('2010-12-10 00:00:00'), Timestamp('2011-03-13 03:00:00'), Timestamp('2011-11-06 02:00:00'), Timestamp('2012-03-11 03:00:00'), Timestamp('2012-11-04 02:00:00'), Timestamp('2012-12-06 04:00:00'), Timestamp('2013-03-10 03:00:00'), Timestamp('2013-11-03 02:00:00'), Timestamp('2014-03-09 03:00:00'), Timestamp('2014-03-11 14:00:00'), Timestamp('2015-03-08 03:00:00'), Timestamp('2016-03-13 03:00:00'), Timestamp('2017-03-12 03:00:00'), Timestamp('2018-03-11 03:00:00')]


In [21]:
df = df.resample('h').first().fillna(np.nan)

missing_Timestamp = pd.date_range(
    '2004-10-01',
    '2018-08-03',
    freq='h'
).difference(df.index)

print('\nNumber of Missing Timestamp= ', len(missing_Timestamp), '\n')
print(missing_Timestamp.to_list())


Number of Missing Timestamp=  1 

[Timestamp('2004-10-01 00:00:00')]


In [22]:
df.reset_index(inplace=True)
len_list=df[df['AEP_MW'].isnull()].index.tolist()
df.iloc[len_list]

,Datetime,AEP_MW
721,2004-10-31 02:00:00,NaN
4418,2005-04-03 03:00:00,NaN
9457,2005-10-30 02:00:00,NaN
13154,2006-04-02 03:00:00,NaN
18193,2006-10-29 02:00:00,NaN
21386,2007-03-11 03:00:00,NaN
27097,2007-11-04 02:00:00,NaN
30122,2008-03-09 03:00:00,NaN
35833,2008-11-02 02:00:00,NaN
38858,2009-03-08 03:00:00,NaN


In [23]:
df.head()

,Datetime,AEP_MW
0,2004-10-01 01:00:00,12379.0
1,2004-10-01 02:00:00,11935.0
2,2004-10-01 03:00:00,11692.0
3,2004-10-01 04:00:00,11597.0
4,2004-10-01 05:00:00,11681.0


In [24]:
df.isnull().sum()

Datetime     0
AEP_MW      27
dtype: int64

In [25]:
df['AEP_MW'] = df['AEP_MW'].interpolate()

In [26]:
df.isnull().sum()

Datetime    0
AEP_MW      0
dtype: int64

In [27]:
df.to_csv(r'Z:\University\8th Semester\ML Lab\Lab 3\2_Missing_Values_Filled.csv',index=False)

In [28]:
print('\tSummary of American Electric Power (AEP)')
print('\nStart Date: \n\t',df.head(1))
print('\nEnd Date: \n\t',  df.tail(1))
print('\nLengthBF: 121273')
print('\nLengthAF: ',len(df))
print('\nSamplesBF: 5053')
print('\nSamplesAF: ',(len(df)/24))
print('\nMissing Points: ',len(len_list))
print('\nMissing Points are at indices:\n ',len_list)

	Summary of American Electric Power (AEP)

Start Date: 
	              Datetime   AEP_MW
0 2004-10-01 01:00:00  12379.0

End Date: 
	          Datetime   AEP_MW
121295 2018-08-03  14809.0

LengthBF: 121273

LengthAF:  121296

SamplesBF: 5053

SamplesAF:  5054.0

Missing Points:  27

Missing Points are at indices:
  [721, 4418, 9457, 13154, 18193, 21386, 27097, 30122, 35833, 38858, 44569, 47762, 53473, 54263, 56498, 62209, 65234, 70945, 71715, 73970, 79681, 82706, 82765, 91442, 100346, 109082, 117818]


## Final Conclusion
In this lab, the AEP hourly dataset was imported and inspected. The analysis helped identify the structure of the time-series data and prepared it for missing value treatment in the next lab.